# Limpeza de Dados: The Movies Dataset
Esta etapa realiza a **higienização completa** do `movies_metadata.csv`, um dataset de alta heterogeneidade com colunas em formato JSON aninhado, campos financeiros corrompidos e registros sem atividade de público. O produto desta etapa é um dataset tabular, tipado e livre de ruídos — pronto para receber a camada de inteligência do notebook de Engenharia de Features.

- **Escopo:** Descarte de colunas de baixa utilidade analítica, tratamento de nulos por limiar percentual, filtragem de registros sem engajamento e correção de tipos
- **Produto desta etapa:** `movies_cleaned.csv`

In [1]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast

# Nossos módulos customizados da pasta src/
from src import load_data
from src import save_dataset

--- 
## Carregando Dados

In [2]:
caminho = '../data/raw/movies_dataset/movies_metadata.csv'
df_movies = load_data(caminho, tipo_arquivo='csv')

Dados CSV carregados! Formato: (45466, 24)


---
## Aplicando Limpeza de Missing Data 

### Descarte por Alta Cardinalidade e Ausência Estrutural

Colunas como `homepage` e `tagline` combinam dois problemas que as tornam inviáveis para análise: alta cardinalidade de texto livre e taxa de preenchimento abaixo de 50%. Além de não contribuírem para nenhuma dimensão analítica do projeto, sua presença introduziria ruído nas etapas de modelagem e agregação.

- **Critério:** Colunas com ≥ 50% de nulos **e** sem valor analítico para os objetivos do projeto foram descartadas
- **Princípio:** A redução de dimensionalidade nesta etapa protege a integridade de todas as análises posteriores

In [3]:
# Dropping colunas irrelevantes para a análise que possuem alta cardinalidade de texto
# Essas colunas não são úteis para a análise e podem introduzir ruído
# Além disso, possuem 50% ou mais de valores ausentes, o que pode prejudicar a qualidade dos dados

colunas_para_dropar = ['homepage', 'tagline', 'overview','poster_path', 'status', 'imdb_id']

df_movies_limpo = df_movies.drop(columns=colunas_para_dropar)

In [4]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   id                     45466 non-null  str    
 5   original_language      45455 non-null  str    
 6   original_title         45466 non-null  str    
 7   popularity             45461 non-null  str    
 8   production_companies   45463 non-null  str    
 9   production_countries   45463 non-null  str    
 10  release_date           45379 non-null  str    
 11  revenue                45460 non-null  float64
 12  runtime                45203 non-null  float64
 13  spoken_languages       45460 non-null  str    
 14  title                  45460 non-null  str    
 15  video        

### Filtro de Engajamento de Público: Exclusão de Filmes sem Audiência Verificável

Filmes com zero votos não possuem sinal de recepção do público — são registros incompletos que distorceriam qualquer métrica de popularidade, média de avaliação ou análise de correlação entre orçamento e aprovação. A presença desses registros invalidaria conclusões sobre o comportamento real do mercado cinematográfico.

In [5]:
# Dropando filmes com 0 votos
df_movies_limpo = df_movies_limpo[df_movies_limpo['vote_count'] > 0]

In [6]:
numero_filmes_zero_votos = (df_movies_limpo['vote_count'] == 0).sum()
print(f'Número de filmes com 0 votos: {numero_filmes_zero_votos}')

Número de filmes com 0 votos: 0


### Integridade da Dimensão Temporal do Filme

A duração (`runtime`) é uma variável de segmentação relevante para distinguir longas-metragens de curtas e para análises comparativas de gênero. Registros com duração ausente ou igual a zero representam cadastros incompletos que não podem ser classificados em nenhuma categoria de produto cinematográfico.

In [7]:
# Dropando coluna runtime
df_movies_limpo = df_movies_limpo.dropna(subset=['runtime'])
df_movies_limpo = df_movies_limpo[df_movies_limpo['runtime'] > 0]

In [8]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41190 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  41190 non-null  str    
 1   belongs_to_collection  4344 non-null   str    
 2   budget                 41190 non-null  str    
 3   genres                 41190 non-null  str    
 4   id                     41190 non-null  str    
 5   original_language      41184 non-null  str    
 6   original_title         41190 non-null  str    
 7   popularity             41190 non-null  str    
 8   production_companies   41190 non-null  str    
 9   production_countries   41190 non-null  str    
 10  release_date           41170 non-null  str    
 11  revenue                41190 non-null  float64
 12  runtime                41190 non-null  float64
 13  spoken_languages       41190 non-null  str    
 14  title                  41190 non-null  str    
 15  video             

### Estratégia de Tratamento de Nulos por Limiar Percentual

Para as colunas restantes, a decisão de descartar linhas versus manter registros com nulos não é arbitrária: ela é orientada pelo **impacto percentual sobre o volume total do dataset**. Colunas com taxa de ausência ≤ 0,1% têm seus registros nulos removidos sem impacto estatístico relevante, preservando a qualidade sem sacrificar volume analítico significativo.

- **Critério operacional:** Perda de dados aceitável definida em ≤ 0,1% do dataset total
- **Resultado esperado:** Máxima completude de registros nas colunas analíticas críticas

In [9]:
# Calculando os nulos de todas as colunas
nulos_por_coluna = df_movies.isnull().sum()

# Filtramndopara manter APENAS as colunas que têm buracos (> 0)
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0]

# Calculamos a porcentagem
percentual_nulos = (colunas_com_nulos / len(df_movies)) * 100

# Juntando tudo num DataFrame de resumo
df_resumo_nulos = pd.DataFrame({
    'Valores Nulos (Qtd)': colunas_com_nulos,
    'Perda de Dados (%)': percentual_nulos
}).sort_values(by='Perda de Dados (%)', ascending=False)

# Selecionando apenas as colunas com perda de dados menor que 0.1% para considerar o drop
df_resumo_nulos_para_dropar = df_resumo_nulos[df_resumo_nulos['Perda de Dados (%)'] < 1]

In [10]:
# Dropando as linhas com menos de 1% de perda de dados

# Criando uma lista de colunas que tem perda de dados menor ou igual a 1%
indices_limpos = [i.split(' (')[0] for i in df_resumo_nulos_para_dropar.index]

# Criando uma lista segura de colunas para drop, garantindo que elas existam no DataFrame
lista_segura = [coluna for coluna in indices_limpos if coluna in df_movies_limpo.columns]

# Dropando as linhas com menos de 0.1% de perda de dados
df_movies_limpo = df_movies_limpo.dropna(subset=lista_segura)

In [11]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41164 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  41164 non-null  str    
 1   belongs_to_collection  4344 non-null   str    
 2   budget                 41164 non-null  str    
 3   genres                 41164 non-null  str    
 4   id                     41164 non-null  str    
 5   original_language      41164 non-null  str    
 6   original_title         41164 non-null  str    
 7   popularity             41164 non-null  str    
 8   production_companies   41164 non-null  str    
 9   production_countries   41164 non-null  str    
 10  release_date           41164 non-null  str    
 11  revenue                41164 non-null  float64
 12  runtime                41164 non-null  float64
 13  spoken_languages       41164 non-null  str    
 14  title                  41164 non-null  str    
 15  video             

In [12]:
display(f"Número de linhas após limpeza: {len(df_movies_limpo)}")
display(f"Número de linhas removidas: {len(df_movies) - len(df_movies_limpo)}")

'Número de linhas após limpeza: 41164'

'Número de linhas removidas: 4302'

### Correção de Tipos: Habilitando Análises Financeiras e Temporais

As colunas financeiras e de data chegam do CSV como strings — um efeito colateral da heterogeneidade do dataset original. Sem a conversão para seus tipos nativos, qualquer operação de ordenação temporal, cálculo de ROI ou agregação financeira produziria resultados incorretos ou erros silenciosos.

In [13]:
# Transformando a coluna 'release_date' para datetime
df_movies_limpo['release_date'] = pd.to_datetime(df_movies_limpo['release_date'], errors='coerce')

# Forçando a conversão para numérico (float)
df_movies_limpo['budget'] = pd.to_numeric(df_movies_limpo['budget'], errors='coerce')
df_movies_limpo['popularity'] = pd.to_numeric(df_movies_limpo['popularity'], errors='coerce')

# Tranfromando as colunas 'adult' e 'video' para booleano
df_movies_limpo['adult'] = df_movies_limpo['adult'].astype(str).str.strip().str.lower() == 'true'
df_movies_limpo['video'] = df_movies_limpo['video'].astype(str).str.strip().str.lower() == 'true'

# Verificando o resultado final
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41164 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   adult                  41164 non-null  bool          
 1   belongs_to_collection  4344 non-null   str           
 2   budget                 41164 non-null  int64         
 3   genres                 41164 non-null  str           
 4   id                     41164 non-null  str           
 5   original_language      41164 non-null  str           
 6   original_title         41164 non-null  str           
 7   popularity             41164 non-null  float64       
 8   production_companies   41164 non-null  str           
 9   production_countries   41164 non-null  str           
 10  release_date           41164 non-null  datetime64[us]
 11  revenue                41164 non-null  float64       
 12  runtime                41164 non-null  float64       
 13  spoken_languages 

### Tratamento de Zeros como Dados Ausentes: Proteção da Integridade Financeira

No contexto deste dataset, valores iguais a zero nas colunas `budget` e `revenue` não representam produções sem custo ou sem receita — representam **ausência de dado declarado**. Mantê-los como zero distorceria qualquer análise de distribuição financeira, cálculo de ROI médio ou correlação entre investimento e desempenho de bilheteria.

- **Regra de negócio:** Zero financeiro neste dataset equivale semanticamente a `NaN` — campo não informado
- **Impacto:** Preserva a validade estatística de todas as métricas financeiras nas análises subsequentes

In [14]:
# Mudando 0 para NaN para não distorcer as análises de orçamento e receita
df_movies_limpo['budget'] = df_movies_limpo['budget'].replace(0, np.nan) 
df_movies_limpo['revenue'] = df_movies_limpo['revenue'].replace(0, np.nan) 

In [15]:
display(
    df_movies_limpo.head(5)
    .style
    .set_caption("Movies Dataset — Pós-Limpeza")
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#2B2D42'), ('color', 'white'), ('text-align', 'left')]
    }])
    .map(lambda v: 'color: #888888; font-style: italic' if pd.isna(v) else '')
    .format(
        na_rep='—',
        formatter={
            'budget': 'US$ {:,.0f}',
            'revenue': 'US$ {:,.0f}',
            'vote_average': '{:.1f}',
        }
    )
)

,adult,belongs_to_collection,budget,genres,id,original_language,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}","US$ 30,000,000","[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",862,en,Toy Story,21.946943,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-10-30 00:00:00,"US$ 373,554,033",81.000000,"[{'iso_639_1': 'en', 'name': 'English'}]",Toy Story,False,7.7,5415.000000
1,False,—,"US$ 65,000,000","[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",8844,en,Jumanji,17.015539,"[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications', 'id': 10201}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-15 00:00:00,"US$ 262,797,249",104.000000,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Jumanji,False,6.9,2413.000000
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'poster_path': '/nLvUdqgPgm3F85NMCii9gVFUcet.jpg', 'backdrop_path': '/hypTnLot2z8wpFS7qwsQHW1uV8u.jpg'}",—,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",15602,en,Grumpier Old Men,11.712900,"[{'name': 'Warner Bros.', 'id': 6194}, {'name': 'Lancaster Gate', 'id': 19464}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-22 00:00:00,—,101.000000,"[{'iso_639_1': 'en', 'name': 'English'}]",Grumpier Old Men,False,6.5,92.000000
3,False,—,"US$ 16,000,000","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]",31357,en,Waiting to Exhale,3.859495,"[{'name': 'Twentieth Century Fox Film Corporation', 'id': 306}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-22 00:00:00,"US$ 81,452,156",127.000000,"[{'iso_639_1': 'en', 'name': 'English'}]",Waiting to Exhale,False,6.1,34.000000
4,False,"{'id': 96871, 'name': 'Father of the Bride Collection', 'poster_path': '/nts4iOmNnq7GNicycMJ9pSAn204.jpg', 'backdrop_path': '/7qwE57OVZmMJChBpLEbJEmzUydk.jpg'}",—,"[{'id': 35, 'name': 'Comedy'}]",11862,en,Father of the Bride Part II,8.387519,"[{'name': 'Sandollar Productions', 'id': 5842}, {'name': 'Touchstone Pictures', 'id': 9195}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-02-10 00:00:00,"US$ 76,578,911",106.000000,"[{'iso_639_1': 'en', 'name': 'English'}]",Father of the Bride Part II,False,5.7,173.000000


---
## Deserialização de Colunas Estruturadas: De JSON Serializado para Listas Nativas

Colunas como `genres`, `production_countries`, `spoken_languages` e `belongs_to_collection` chegam do CSV como strings que representam estruturas JSON — um formato que preserva a estrutura original, mas torna os dados **opacos para qualquer operação analítica**. Sem a extração dos valores internos, é impossível filtrar por gênero, agrupar por país de produção ou identificar pertencimento a franquias.

- **Decisão arquitetural:** A extração é centralizada em funções reutilizáveis para garantir tratamento uniforme e seguro de nulos em todas as colunas afetadas
- **Resultado:** Cada coluna passa a armazenar uma lista Python nativa, viabilizando operações de explode, filtragem e contagem nas análises subsequentes

In [16]:
"""
Extrai o valor da chave 'name' de uma string no formato lista de dicionários.
Se o valor for nulo, vazio ou corrompido, retorna uma lista com o valor padrão.
"""
def extrair_nomes(texto, valor_padrao="Não Identificado"):

    # Trata os vazios de sistema (NaN, None)
    if pd.isna(texto):
        return [valor_padrao]
    
    try:
        # ast.literal_eval converte a string em lista Python real.
        # Adicionei uma proteção: se por acaso já for uma lista, ele não quebra.
        lista = ast.literal_eval(texto) if isinstance(texto, str) else texto
        
        # Trata as listas que vieram vazias da API (ex: "[]" que virou [])
        if not lista or len(lista) == 0:
            return [valor_padrao]
        
        # Extrai apenas o 'name' de cada dicionário (O Caminho Feliz)
        return [item.get('name', valor_padrao) for item in lista]
    
    except (ValueError, SyntaxError, TypeError):
        # Se a string estiver corrompida e o ast falhar, salva a linha com o padrão
        return [valor_padrao]

In [17]:
# Aplicando a função mágica para cada coluna com seu respectivo nome padrão!
df_movies_limpo['genres'] = df_movies_limpo['genres'].apply(
    lambda x: extrair_nomes(x, valor_padrao="Gênero Não Identificado")
)

df_movies_limpo['production_companies'] = df_movies_limpo['production_companies'].apply(
    lambda x: extrair_nomes(x, valor_padrao="Companhia Não Identificada")
)

df_movies_limpo['production_countries'] = df_movies_limpo['production_countries'].apply(
    lambda x: extrair_nomes(x, valor_padrao="País Não Identificado")
)

df_movies_limpo['spoken_languages'] = df_movies_limpo['spoken_languages'].apply(
    lambda x: extrair_nomes(x, valor_padrao="Idioma Não Identificado")
)

### Extração de Franquias (Coleções)

A coluna `belongs_to_collection` difere das demais por armazenar um único dicionário (JSON) em vez de uma lista. 

- **Objetivo:** Extrair o nome da franquia à qual o filme pertence.
- **Regra de Negócio:** Diferente de gêneros ou produtoras, a ausência de dados nesta coluna não indica erro na base, mas sim que a obra é um **Filme Isolado (Standalone)**. A função foi adaptada para retornar o tipo `String` uniformemente e imputar o valor "Filme Isolado" para os valores nulos, permitindo futuras análises comparativas de bilheteria entre sagas vs. filmes originais.

In [18]:
"""
Extrai o nome da franquia de um dicionário único.
Garante o retorno de uma String em 100% dos casos.
"""
def extrair_colecao(texto, valor_padrao="Filme Isolado"):

    # Trata nulos retornando STRING (e não lista)
    if pd.isna(texto):
        return valor_padrao
    
    try:
        # Converte a string para dicionário
        dicionario = ast.literal_eval(texto) if isinstance(texto, str) else texto

        return dicionario.get('name', valor_padrao)
    
    except (ValueError, SyntaxError, TypeError, AttributeError):
        # Em caso de string corrompida, retorna a STRING padrão
        return valor_padrao

In [19]:
# Aplicando a função na base de dados
df_movies_limpo['belongs_to_collection'] = df_movies_limpo['belongs_to_collection'].apply(
    lambda x: extrair_colecao(x))

In [20]:
# Validação pós-deserialização: confirmar que as colunas agora armazenam listas nativas
lista_colunas = [
    'original_title', 
    'belongs_to_collection',
    'genres',
    'production_companies', 
    'production_countries', 
    'spoken_languages'
]

display (
    df_movies_limpo[lista_colunas]
    .head(10)
    .style
    .set_properties(
        subset=['original_title'],
        **{'background-color': '#2B2D42', 'color': 'white'}
    )
    .set_properties(
        subset=['belongs_to_collection'],
        **{'background-color': '#9A8C98', 'color': 'black'} 
    )
    .set_properties(
        subset=['genres'],
        **{'background-color': '#6D597A', 'color': 'white'}
    )
    .set_properties(
        subset=['production_companies'],
        **{'background-color': '#4A4E69', 'color': 'white'} 
    )
    .set_properties(
        subset=['production_countries'],
        **{'background-color': '#588157', 'color': 'white'}
    )
    .set_properties(
        subset=['spoken_languages'],
        **{'background-color': '#3D5A80', 'color': 'white'}
    )
)

,original_title,belongs_to_collection,genres,production_companies,production_countries,spoken_languages
0,Toy Story,Toy Story Collection,"['Animation', 'Comedy', 'Family']",['Pixar Animation Studios'],['United States of America'],['English']
1,Jumanji,Filme Isolado,"['Adventure', 'Fantasy', 'Family']","['TriStar Pictures', 'Teitler Film', 'Interscope Communications']",['United States of America'],"['English', 'Français']"
2,Grumpier Old Men,Grumpy Old Men Collection,"['Romance', 'Comedy']","['Warner Bros.', 'Lancaster Gate']",['United States of America'],['English']
3,Waiting to Exhale,Filme Isolado,"['Comedy', 'Drama', 'Romance']",['Twentieth Century Fox Film Corporation'],['United States of America'],['English']
4,Father of the Bride Part II,Father of the Bride Collection,['Comedy'],"['Sandollar Productions', 'Touchstone Pictures']",['United States of America'],['English']
5,Heat,Filme Isolado,"['Action', 'Crime', 'Drama', 'Thriller']","['Regency Enterprises', 'Forward Pass', 'Warner Bros.']",['United States of America'],"['English', 'Español']"
6,Sabrina,Filme Isolado,"['Comedy', 'Romance']","['Paramount Pictures', 'Scott Rudin Productions', 'Mirage Enterprises', 'Sandollar Productions', 'Constellation Entertainment', 'Worldwide', 'Mont Blanc Entertainment GmbH']","['Germany', 'United States of America']","['Français', 'English']"
7,Tom and Huck,Filme Isolado,"['Action', 'Adventure', 'Drama', 'Family']",['Walt Disney Pictures'],['United States of America'],"['English', 'Deutsch']"
8,Sudden Death,Filme Isolado,"['Action', 'Adventure', 'Thriller']","['Universal Pictures', 'Imperial Entertainment', 'Signature Entertainment']",['United States of America'],['English']
9,GoldenEye,James Bond Collection,"['Adventure', 'Action', 'Thriller']","['United Artists', 'Eon Productions']","['United Kingdom', 'United States of America']","['English', 'Pусский', 'Español']"


---
## Salvando Dataset Limpo 


In [21]:
# Salvando o dataset limpo para a próxima etapa de análise
# Salvando em CSV para visualização e compartilhamento com ferramentas como PI
# Salvando em PKL para uso na engenharia de freacture e presenrvação de tipos 
save_dataset(df=df_movies_limpo, nome_arquivo='movies_cleaned', tipo_arquivo='both')

Sucesso! Ficheiro guardado em '..\data\processed\movies_cleaned.pkl' e '..\data\processed\movies_cleaned.csv' 


---
## Conclusão da Limpeza e Estruturação de Dados (É necessario atualizar markdown)

O dataset de filmes passou pelas etapas essenciais de higienização e formatação para garantir a integridade das análises financeiras e de recepção do público. As principais intervenções nesta etapa incluíram:
* **Descarte de colunas de alta cardinalidade de texto**
    - Colunas descartadas: `homepage`, `poster_path`, `overview`, `tagline`, `status` e `imdb_id`.
* **Tratamento de Inconsistências:** Identificação e manejo de valores nulos e registros discrepantes, com atenção especial às métricas críticas de negócio, como `budget` (orçamento) e `revenue` (receita). 
* **Padronização e Tipagem (Data Casting):** Conversão de formatos genéricos para tipos otimizados. Isso incluiu a transformação de indicadores binários para booleanos, estruturação de datas de lançamento para `datetime`, habilitando futuras análises de séries temporais e strings que representam JSON para listas nativas do python.

**Exportação:**
O dataset resultante, agora estruturalmente validado, foi exportado com sucesso para `data/processed/movies_cleaned.csv`.

**Próximo Passo:**
Com a base de filmes íntegra e padronizada, o pipeline avança para a **Engenharia de Features** (para criação de variáveis como lucro e extração de dicionários JSON) e, na sequência, para a **Análise Exploratória de Dados (EDA)**, onde investigaremos as dinâmicas de bilheteria e engajamento.

* **Vá para o notebook:** `08_feature_engineering_movies.ipynb`.
---